In [ ]:
# https://datalake.viettelcyber.com/gateway/ui/zeppelin/#/notebook/2MEFF3M9D

In [ ]:
%livy.pyspark
from pyspark import StorageLevel
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import time
from datetime import datetime

backup_folder_name ="fact_cso_tickets/" + datetime.now().strftime("%Y/%m/%d/%H%M%S")

df_parquet = spark.read.parquet("/opt/datasets/crawlers/vcs/freshdesk/data/fact_cso_tickets/*")

df_parquet_cache = df_parquet.repartition(10).persist(StorageLevel.MEMORY_AND_DISK)

df_parquet_cache.write \
    .mode("overwrite") \
    .parquet("/opt/datasets/crawlers/vcs/freshdesk/data_parquet/" + backup_folder_name)

w = Window.partitionBy("id").orderBy(F.col("updated_at_ts").desc())

df_latest = (
    df_parquet_cache.withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

df_latest.repartition(10).write \
    .mode("overwrite") \
    .parquet("/opt/datasets/crawlers/vcs/freshdesk/data/fact_cso_tickets")
    
df_parquet.unpersist(blocking=True)
df_parquet = None

df_parquet_cache.unpersist(blocking=True)
df_parquet_cache = None

df_latest.unpersist(blocking=True)
df_latest = None


In [ ]:
%livy.pyspark
from pyspark import StorageLevel
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import time
from datetime import datetime

backup_folder_name ="deals/" + datetime.now().strftime("%Y/%m/%d/%H%M%S")

df_parquet = spark.read.parquet("/opt/datasets/crawlers/vcs/freshworks/data/deals/*")
# df_parquet = spark.read.parquet("/opt/datasets/crawlers/vcs/freshworks/data_parquet/deals/2026/01/05/125729/*")

df_parquet_cache = df_parquet.repartition(10).persist(StorageLevel.MEMORY_AND_DISK)

df_parquet_cache.write \
    .mode("overwrite") \
    .parquet("/opt/datasets/crawlers/vcs/freshworks/data_parquet/" + backup_folder_name)

w = Window.partitionBy("id").orderBy(F.col("updated_at_ts").desc())

df_latest = (
    df_parquet_cache.withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

df_latest.repartition(10).write \
    .mode("overwrite") \
    .parquet("/opt/datasets/crawlers/vcs/freshworks/data/deals")
    
df_parquet.unpersist(blocking=True)
df_parquet = None

df_parquet_cache.unpersist(blocking=True)
df_parquet_cache = None

df_latest.unpersist(blocking=True)
df_latest = None
